In [5]:
import json
import glob
import statistics

base_path = "/mnt/dataset4/yinuo/Rest_tuning/result/LaBraM/lora_mask_0.7/wo_label_channel/Multitask_time_cost_test/train_0.7/m3cv/lora"

# 获取所有subject的metrics.json文件
subject_dirs = sorted(glob.glob(f"{base_path}/subject_*/metrics.json"))

best_epochs = []
subject_epochs = []

for filepath in subject_dirs:
    with open(filepath, 'r') as f:
        data = json.load(f)
    subject_id = data.get('subject_id', 'unknown')
    best_epoch = data.get('best_epoch')
    if best_epoch is not None:
        best_epochs.append(best_epoch)
        subject_epochs.append((subject_id, best_epoch))

# 计算均值和方差
n = len(best_epochs)
mean_val = sum(best_epochs) / n
var_val = sum((x - mean_val) ** 2 for x in best_epochs) / n

print(f"总共统计了 {n} 个subject")
print(f"best epoch 列表: {best_epochs}")
print(f"均值 (Mean):   {mean_val:.4f}")
print(f"方差 (Var):    {var_val:.4f}")
print(f"标准差 (Std):  {var_val ** 0.5:.4f}")
print(f"中位数 (Med):  {statistics.median(best_epochs):.4f}")
print(f"最小值 (Min):  {min(best_epochs)}")
print(f"最大值 (Max):  {max(best_epochs)}")

总共统计了 95 个subject
best epoch 列表: [13, 8, 56, 17, 31, 21, 30, 15, 17, 10, 15, 93, 27, 64, 43, 30, 31, 26, 8, 29, 28, 35, 37, 6, 20, 31, 54, 25, 26, 38, 21, 40, 12, 41, 40, 8, 42, 23, 35, 42, 74, 34, 56, 28, 38, 11, 21, 79, 29, 42, 11, 29, 87, 11, 21, 13, 57, 16, 62, 14, 8, 23, 5, 17, 11, 20, 20, 7, 53, 9, 64, 11, 42, 31, 10, 23, 9, 19, 30, 30, 11, 36, 15, 11, 17, 8, 26, 7, 47, 7, 49, 25, 16, 39, 6]
均值 (Mean):   28.2421
方差 (Var):    355.4466
标准差 (Std):  18.8533
中位数 (Med):  25.0000
最小值 (Min):  5
最大值 (Max):  93


In [6]:
import json
import os
import argparse
import statistics

def calculate_avg_test_acc(base_dir, subjects=None):
    """
    统计指定被试的平均 test_acc
    
    Args:
        base_dir: 包含 subject_{id} 文件夹的目录
        subjects: 指定的 subject id 列表，如果为 None 则统计所有存在的 subject
    """
    if subjects is None:
        # 自动发现所有 subject 文件夹
        subjects = []
        for d in sorted(os.listdir(base_dir)):
            if d.startswith('subject_') and os.path.isdir(os.path.join(base_dir, d)):
                try:
                    sid = int(d.split('_')[1])
                    subjects.append(sid)
                except ValueError:
                    continue
        subjects.sort()
    
    test_accs = []
    results = []
    
    for sid in subjects:
        subject_dir = os.path.join(base_dir, f"subject_{sid:02d}")
        metrics_path = os.path.join(subject_dir, "metrics.json")
        
        if not os.path.exists(metrics_path):
            print(f"  ⚠️  Subject {sid:02d}: metrics.json not found, skipped")
            continue
        
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        
        test_acc = metrics.get('test_acc', None)
        if test_acc is not None:
            test_accs.append(test_acc)
            results.append({
                'subject_id': sid,
                'test_acc': test_acc,
                'test_f1': metrics.get('test_f1', None),
                'best_val_acc': metrics.get('best_val_acc', None),
                'best_epoch': metrics.get('best_epoch', None)
            })
            print(f"  ✓ Subject {sid:02d}: test_acc = {test_acc:.4f}")
        else:
            print(f"  ⚠️  Subject {sid:02d}: test_acc not found in metrics.json")
    
    print("\n" + "="*60)
    print(f"Total subjects processed: {len(results)}")
    
    if len(test_accs) > 0:
        mean_acc = statistics.mean(test_accs)
        std_acc = statistics.stdev(test_accs) if len(test_accs) > 1 else 0.0
        print(f"\n📊 Average Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
        print(f"📊 Min Test Accuracy:     {min(test_accs):.4f}")
        print(f"📊 Max Test Accuracy:     {max(test_accs):.4f}")
        print(f"📊 Median Test Accuracy:  {statistics.median(test_accs):.4f}")
    else:
        print("\n✗ No valid test_acc found")
    
    return results, test_accs


def main():
    parser = argparse.ArgumentParser(description='统计指定被试的平均 test_acc')
    parser.add_argument('--base_dir',
                        default='/mnt/dataset4/yinuo/Rest_tuning/result/LaBraM/baseline_continue_pretrain_new/1_15/Multitask/train_0.1/cogbci/linear',
                        type=str,
                        help='包含 subject_{id} 文件夹的目录')
    parser.add_argument('--subjects', type=int, nargs='+', default=None,
                        help='指定一个或多个 subject id (例如: --subjects 1 3 5)')
    
    args = parser.parse_args()
    
    print(f"📁 Base directory: {args.base_dir}")
    if args.subjects:
        print(f"🎯 Specified subjects: {args.subjects}")
    else:
        print("🎯 Mode: All available subjects")
    print("="*60)
    
    calculate_avg_test_acc(args.base_dir, args.subjects)

In [43]:
import json
import glob
import numpy as np
from pathlib import Path

# 设置搜索路径（自动匹配所有 subject_* 子目录）
base_dir = Path("/mnt/dataset4/yinuo/Rest_tuning/result/BIOT/Baseline/Multitask_time_cost_test_2/train_0.7/cogbci/full")

# 查找所有 metrics.json
pattern = str(base_dir / "subject_*" / "metrics.json")
files = sorted(glob.glob(pattern))

# print(f"找到 {len(files)} 个 metrics.json 文件\n")

train_times = []
invalid_files = []

for f in files:
    try:
        with open(f, 'r', encoding='utf-8') as fp:
            data = json.load(fp)
        
        t = data.get('train_time_sec')
        if t is not None:
            train_times.append(t)
            subj = Path(f).parent.name
            # print(f"{subj}: train_time_sec = {t:.3f}s")
        else:
            print(f"⚠️ {f}: 未找到 train_time_sec")
            invalid_files.append(f)
    except Exception as e:
        print(f"✗ {f}: {e}")
        invalid_files.append(f)

# 统计结果
if train_times:
    arr = np.array(train_times)
    print(f"\n{'='*50}")
    print(f"有效样本数:   {len(arr)}")
    print(f"平均值:       {arr.mean():.3f} 秒")
    print(f"标准差:       {arr.std(ddof=1):.3f} 秒")   # 样本标准差
    print(f"方差:         {arr.var(ddof=1):.6f} 秒²")  # 样本方差
    print(f"最小值:       {arr.min():.3f} 秒")
    print(f"最大值:       {arr.max():.3f} 秒")
    print(f"中位数:       {np.median(arr):.3f} 秒")
else:
    print("\n未找到有效的 train_time_sec 数据")


有效样本数:   26
平均值:       403.206 秒
标准差:       109.449 秒
方差:         11979.047854 秒²
最小值:       264.010 秒
最大值:       737.034 秒
中位数:       381.294 秒


<function __main__.main()>